# BR03A — Audit retrieval paths and prompt evidence for leakage

Run **after BR03 and before generation**, in the CPU/graph environment, separately for `internal_val`, `internal_test` and `shaoxing` as needed. This reads the completed banks, paths and packets and writes separate audits. It does not change references, drop queries, load a generator or parse/use query target labels.

The audit reconstructs the frozen graph from **training candidates only**, checks graph counts and patient splits, validates training-bank identities, excludes every Symile patient from the MIMIC-CXR references, and checks forward/reverse patient exclusions. It reconstructs every prompt case from approved reference findings/labs and frozen graph support. Changed/extra packet evidence stops the gate; no automatic repair or exclusion occurs.

This is an additional layer over BR00–BR03. **BR01B can continue unchanged**: none of its frozen Python files/configuration is replaced. Cross-source waveform similarity, near-duplicate images, encoder training history and foundation-model pretraining are not certified by this audit.

In [ ]:
from pathlib import Path
import sys
EXP = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / 'config.example.json').is_file() and (p / 'common.py').is_file()), None)
if EXP is None: raise FileNotFoundError('Start Jupyter inside graph_reasoning/experiment02')
sys.path.insert(0, str(EXP))
from common import load_config, parent_setup, read
CONFIG_PATH = EXP / 'config.json'
if not CONFIG_PATH.is_file():
    raise FileNotFoundError('Run BR00 first to create experiment02/config.json')
CFG = load_config(CONFIG_PATH)
PROJECT = parent_setup(CFG)
RUN = Path(CFG['run_dir'])
print('Kernel:', sys.executable)
print('Cohort:', CFG['cohort'], '| outputs:', RUN)


In [ ]:
from extensions.leakage_controls import audit_packets
MAX_RUN_HOURS = 1.0  # Below allocation time remaining; the audit needs no GPU.
summary = audit_packets(CFG, max_run_hours=MAX_RUN_HOURS)
AUDIT = RUN / 'audits' / 'leakage_controls' / CFG['cohort']
print(summary)
print('Summary:', AUDIT / 'summary.json')
print('Per-query table:', AUDIT / 'queries.csv')
if not summary['complete'] or not summary['passed']:
    raise RuntimeError('Leakage audit is not green. Inspect per-query JSON files; do not start generation.')
print('PASS: continue to BR04A_generate_reports_with_leakage_gate.ipynb')

Each `queries/<query_id>.json` records anchor admissions/patients, candidate studies/patients, selected IDs, reverse admissions, packet/retrieval/graph hashes and the approved case hashes. The CSV is a compact pass/fail table. These are private research audit artifacts containing patient identifiers.

On interruption, rerun the audit cell. Completed per-query files remain on disk, but every input and query is checked again to avoid trusting stale green checks. `audits/leakage_controls/<cohort>/STOP` stops between queries; remove it before resuming. The summary is revoked at the beginning of every audit and remains non-passing if input verification fails.

Use validation data for k/prompt/threshold decisions, then freeze settings before external evaluation. Previously inspected test results remain exploratory. Keep independent clinical targets and reviewer linkage outside all generator contexts.